# 제20장: AI 거버넌스의 미래

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch20.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**AGI 거버넌스 준비도 평가 프레임워크**

In [ ]:
class AGIGovernanceReadinessEvaluator:
    """
    AGI Governance Readiness Assessment
    (Alignment, Oversight, Containment, etc.)
    """
    def __init__(self):
        self.areas = {
            "Alignment": "Value alignment with human intent",
            "Oversight": "Human monitoring capabilities",
            "Containment": "Ability to restrict system scope",
            "Transparency": "Interpretability of reasoning",
            "Accountability": "Clear responsibility chain",
            "Reversibility": "Ability to undo actions",
            "Corrigibility": "System cooperates with correction",
            "Robustness": "Resistance to adversarial attacks",
        }
        self.risk_levels = {
            "existential": {"threshold": 30, "label": "EXISTENTIAL RISK"},
            "catastrophic": {"threshold": 50, "label": "CATASTROPHIC RISK"},
            "severe": {"threshold": 70, "label": "SEVERE RISK"},
            "moderate": {"threshold": 85, "label": "MODERATE RISK"},
            "manageable": {"threshold": 100, "label": "MANAGEABLE"},
        }

    def assess_readiness(self, scores):
        overall = sum(scores.values()) / len(scores)
        critical_gaps = []
        recommendations = []

        for area, desc in self.areas.items():
            score = scores.get(area, 0)
            if score < 50:
                critical_gaps.append(area)
            elif score < 70:
                recommendations.append(
                    f"{area}: Improvement needed ({desc})"
                )

        readiness_level = (
            "NOT READY" if critical_gaps
            else "CONDITIONAL" if recommendations
            else "READY"
        )

        # Determine risk level
        risk_label = "MANAGEABLE"
        for level, info in sorted(
            self.risk_levels.items(),
            key=lambda x: x[1]["threshold"]
        ):
            if overall < info["threshold"]:
                risk_label = info["label"]
                break

        return {
            "overall_score": overall,
            "readiness": readiness_level,
            "risk_level": risk_label,
            "critical_gaps": critical_gaps,
            "recommendations": recommendations,
            "areas_assessed": len(self.areas),
        }

    def generate_improvement_plan(self, scores):
        """Generate prioritized improvement plan"""
        gaps = sorted(scores.items(), key=lambda x: x[1])
        plan = []
        for area, score in gaps:
            if score < 70:
                urgency = "CRITICAL" if score < 50 else "HIGH"
                plan.append({
                    "area": area,
                    "current_score": score,
                    "target_score": 70,
                    "urgency": urgency,
                    "description": self.areas.get(area, ""),
                })
        return plan

**멀티에이전트 감사 추적 시스템**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from datetime import datetime
from enum import Enum
import json
import hashlib

class ActionType(Enum):
    MESSAGE_SENT = "message_sent"
    MESSAGE_RECEIVED = "message_received"
    TOOL_CALLED = "tool_called"
    DECISION_MADE = "decision_made"
    ESCALATION = "escalation"
    DELEGATION = "delegation"
    VETO = "veto"
    ERROR = "error"
    CONSENSUS = "consensus"

@dataclass
class AgentAction:
    timestamp: str
    agent_id: str
    action_type: ActionType
    target_agent: Optional[str]
    payload: Dict[str, Any]
    parent_action_id: Optional[str]  # For causal chain
    action_id: str = ""
    integrity_hash: str = ""

    def __post_init__(self):
        if not self.action_id:
            raw = (f"{self.timestamp}{self.agent_id}"
                   f"{self.action_type.value}")
            self.action_id = hashlib.sha256(
                raw.encode()
            ).hexdigest()[:16]
        self._compute_hash()

    def _compute_hash(self):
        data = json.dumps({
            "timestamp": self.timestamp,
            "agent_id": self.agent_id,
            "action_type": self.action_type.value,
            "payload": str(self.payload),
            "parent": self.parent_action_id,
        }, sort_keys=True)
        self.integrity_hash = hashlib.sha256(
            data.encode()
        ).hexdigest()

### 실행

In [ ]:
class MultiAgentAuditTrail:
    """
    Comprehensive audit trail for multi-agent systems.
    Tracks all inter-agent communications, decisions,
    and tool usage with causal linking.
    """
    def __init__(self):
        self.actions: List[AgentAction] = []
        self.agent_registry: Dict[str, Dict] = {}
        self._action_index: Dict[str, AgentAction] = {}

    def register_agent(self, agent_id: str, role: str,
                       permissions: List[str],
                       owner: str):
        """Register an agent in the governance system"""
        self.agent_registry[agent_id] = {
            "role": role,
            "permissions": permissions,
            "owner": owner,
            "registered_at": datetime.now().isoformat(),
        }

    def log_action(self, agent_id: str,
                   action_type: ActionType,
                   payload: Dict,
                   target_agent: Optional[str] = None,
                   parent_action_id: Optional[str] = None
                   ) -> str:
        """Log an agent action with causal linking"""
        action = AgentAction(
            timestamp=datetime.now().isoformat(),
            agent_id=agent_id,
            action_type=action_type,
            target_agent=target_agent,
            payload=payload,
            parent_action_id=parent_action_id,
        )
        self.actions.append(action)
        self._action_index[action.action_id] = action
        return action.action_id

    def get_causal_chain(self, action_id: str) -> List[AgentAction]:
        """Reconstruct the causal chain for an action"""
        chain = []
        current = self._action_index.get(action_id)
        while current:
            chain.append(current)
            if current.parent_action_id:
                current = self._action_index.get(
                    current.parent_action_id
                )
            else:
                break
        return list(reversed(chain))

    def get_agent_actions(self, agent_id: str,
                          action_type: Optional[ActionType] = None
                          ) -> List[AgentAction]:
        """Get all actions by a specific agent"""
        actions = [
            a for a in self.actions
            if a.agent_id == agent_id
        ]
        if action_type:
            actions = [
                a for a in actions
                if a.action_type == action_type
            ]
        return actions

    def detect_anomalies(self) -> List[Dict]:
        """Detect anomalous patterns in agent behavior"""
        anomalies = []
        from collections import Counter, defaultdict

        # Check for excessive actions by single agent
        agent_counts = Counter(a.agent_id for a in self.actions)
        avg_count = (sum(agent_counts.values())
                     / max(len(agent_counts), 1))
        for agent_id, count in agent_counts.items():
            if count > avg_count * 3:
                anomalies.append({
                    "type": "EXCESSIVE_ACTIVITY",
                    "agent_id": agent_id,
                    "count": count,
                    "average": avg_count,
                })

        # Check for escalation patterns
        escalations = defaultdict(int)
        for a in self.actions:
            if a.action_type == ActionType.ESCALATION:
                escalations[a.agent_id] += 1
        for agent_id, count in escalations.items():
            if count > 5:
                anomalies.append({
                    "type": "FREQUENT_ESCALATION",
                    "agent_id": agent_id,
                    "count": count,
                })

        # Check for unauthorized interactions
        for a in self.actions:
            if (a.target_agent and
                a.target_agent not in self.agent_registry):
                anomalies.append({
                    "type": "UNREGISTERED_TARGET",
                    "agent_id": a.agent_id,
                    "target": a.target_agent,
                })
        return anomalies

    def generate_governance_report(self) -> Dict:
        """Generate comprehensive governance report"""
        total = len(self.actions)
        by_type = Counter(
            a.action_type.value for a in self.actions
        )
        by_agent = Counter(a.agent_id for a in self.actions)
        anomalies = self.detect_anomalies()

        return {
            "report_timestamp": datetime.now().isoformat(),
            "total_actions": total,
            "actions_by_type": dict(by_type),
            "actions_by_agent": dict(by_agent),
            "registered_agents": len(self.agent_registry),
            "anomalies_detected": len(anomalies),
            "anomaly_details": anomalies,
            "integrity_status": "VALID",
        }

**뉴로심볼릭 거버넌스 관리자**

In [ ]:
class NeuroSymbolicGovernanceManager:
    """Enforce symbolic rules on neural network predictions"""
    def __init__(self):
        self.rules = [
            {"id": "R1", "desc": "Minor protection",
             "cond": lambda ctx: ctx.get("age", 99) < 19,
             "action": "BLOCK", "priority": 1},
            {"id": "R2", "desc": "Regulatory limit",
             "cond": lambda ctx: ctx.get("amount", 0) > 1e8,
             "action": "ESCALATE", "priority": 2},
            {"id": "R3", "desc": "Sanctioned entity",
             "cond": lambda ctx: ctx.get("sanctioned", False),
             "action": "BLOCK", "priority": 1},
            {"id": "R4", "desc": "High-risk jurisdiction",
             "cond": lambda ctx: ctx.get("jurisdiction", "")
                     in ["NK", "IR", "SY"],
             "action": "ESCALATE", "priority": 2},
        ]
        self.decision_log = []

    def evaluate(self, neural_pred, confidence, context):
        # Phase 1: Symbolic rule validation (deterministic)
        for rule in sorted(self.rules, key=lambda r: r["priority"]):
            if rule["cond"](context):
                self._log(rule["id"], rule["action"], context)
                return (rule["action"],
                        f"Blocked by Rule {rule['id']}: "
                        f"{rule['desc']}")

        # Phase 2: Confidence check (stochastic)
        if confidence < 0.7:
            self._log("CONF", "HUMAN_REVIEW", context)
            return ("HUMAN_REVIEW",
                    "Low confidence - human review required")

        # Phase 3: Accept neural network result
        self._log("NN", neural_pred, context)
        return neural_pred, "Accepted by Neural Network"

    def verify_rule_consistency(self):
        """Check for rule conflicts"""
        conflicts = []
        for i, r1 in enumerate(self.rules):
            for r2 in self.rules[i+1:]:
                if (r1["action"] != r2["action"] and
                    r1["priority"] == r2["priority"]):
                    conflicts.append({
                        "rule_1": r1["id"],
                        "rule_2": r2["id"],
                        "issue": "Same priority, different actions",
                    })
        return {"consistent": len(conflicts) == 0,
                "conflicts": conflicts}

    def generate_decision_proof(self, context):
        """Generate formal proof of decision"""
        applied_rules = []
        for rule in sorted(self.rules, key=lambda r: r["priority"]):
            result = rule["cond"](context)
            applied_rules.append({
                "rule_id": rule["id"],
                "condition_met": result,
                "action_if_met": rule["action"],
            })
        return {
            "context": context,
            "rules_evaluated": applied_rules,
            "deterministic": True,
            "reproducible": True,
        }

    def _log(self, rule_id, action, context):
        self.decision_log.append({
            "rule": rule_id, "action": action,
            "context_hash": hash(str(context))
        })

**AI-ESG 통합 지표 관리**

In [ ]:
class AIESGGovernance:
    """Collect and report ESG metrics for AI systems"""
    def __init__(self, carbon_budget_tco2e=100.0):
        self.carbon_budget = carbon_budget_tco2e
        self.metrics_history = []

    def report_esg(self, model_name, carbon_tco2e,
                   fairness_score, audit_completed,
                   data_incidents=0, accessibility_score=0.0,
                   jobs_impacted=0):
        budget_remaining = self.carbon_budget - carbon_tco2e
        report = {
            "model": model_name,
            "Environmental": {
                "carbon_footprint": f"{carbon_tco2e:.2f} tCO2e",
                "budget_remaining": f"{budget_remaining:.2f} tCO2e",
                "status": "OK" if budget_remaining > 0 else "EXCEEDED",
                "reduction_target": f"{carbon_tco2e * 0.1:.2f} tCO2e",
            },
            "Social": {
                "fairness_score": f"{fairness_score:.2%}",
                "bias_incidents": data_incidents,
                "accessibility": f"{accessibility_score:.2%}",
                "jobs_impacted": jobs_impacted,
                "status": "OK" if fairness_score > 0.8 else "REVIEW"
            },
            "Governance": {
                "audit_status": "Completed" if audit_completed
                               else "Pending",
                "compliance_gap": not audit_completed,
                "board_reported": audit_completed,
            }
        }
        self.metrics_history.append(report)
        return report

    def compute_carbon_intensity(self, model_name,
                                 total_requests, carbon_tco2e):
        """Carbon per 1000 requests"""
        intensity = (carbon_tco2e / max(total_requests, 1)) * 1000
        return {
            "model": model_name,
            "carbon_per_1k_requests": f"{intensity:.4f} tCO2e",
            "total_requests": total_requests,
            "efficiency_rating":
                "GOOD" if intensity < 0.001
                else "MODERATE" if intensity < 0.01
                else "POOR",
        }

    def generate_annual_esg_report(self):
        """Generate annual ESG report for stakeholders"""
        if not self.metrics_history:
            return {"error": "No data available"}
        total_carbon = sum(
            float(m["Environmental"]["carbon_footprint"]
                  .replace(" tCO2e", ""))
            for m in self.metrics_history
        )
        avg_fairness = sum(
            float(m["Social"]["fairness_score"].replace("%", ""))
            for m in self.metrics_history
        ) / len(self.metrics_history)
        audits_completed = sum(
            1 for m in self.metrics_history
            if m["Governance"]["audit_status"] == "Completed"
        )
        return {
            "period": "Annual",
            "environmental_summary": {
                "total_carbon": f"{total_carbon:.2f} tCO2e",
                "budget_utilization":
                    f"{total_carbon/self.carbon_budget:.0%}",
            },
            "social_summary": {
                "avg_fairness": f"{avg_fairness:.1f}%",
                "total_incidents": sum(
                    m["Social"]["bias_incidents"]
                    for m in self.metrics_history
                ),
            },
            "governance_summary": {
                "audits_completed": audits_completed,
                "audit_rate":
                    f"{audits_completed/len(self.metrics_history):.0%}",
            },
        }

**AI 거버넌스 KPI 대시보드**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from datetime import datetime, timedelta
from enum import Enum

class KPIStatus(Enum):
    ON_TRACK = "on_track"
    AT_RISK = "at_risk"
    OFF_TRACK = "off_track"

@dataclass
class GovernanceKPI:
    name: str
    category: str  # risk, compliance, performance, efficiency
    target: float
    current: float
    unit: str
    direction: str = "higher_is_better"  # or "lower_is_better"

    @property
    def achievement_rate(self) -> float:
        if self.direction == "higher_is_better":
            return self.current / self.target if self.target else 0
        else:
            return self.target / self.current if self.current else 0

    @property
    def status(self) -> KPIStatus:
        rate = self.achievement_rate
        if rate >= 0.9:
            return KPIStatus.ON_TRACK
        elif rate >= 0.7:
            return KPIStatus.AT_RISK
        else:
            return KPIStatus.OFF_TRACK

### 실행

In [ ]:
class GovernanceKPIDashboard:
    """
    PDCA-based AI Governance KPI Dashboard
    Tracks governance health across risk, compliance,
    performance, and efficiency dimensions.
    """
    def __init__(self, organization: str):
        self.organization = organization
        self.kpis: Dict[str, GovernanceKPI] = {}
        self.history: List[Dict] = []
        self.pdca_cycle_count = 0

    def define_kpis(self):
        """Define standard governance KPIs"""
        standard_kpis = [
            GovernanceKPI(
                "AI Incident Rate", "risk",
                target=5.0, current=0.0,
                unit="incidents/quarter",
                direction="lower_is_better"
            ),
            GovernanceKPI(
                "High-Risk AI Ratio", "risk",
                target=20.0, current=0.0,
                unit="%", direction="lower_is_better"
            ),
            GovernanceKPI(
                "Impact Assessment Completion", "compliance",
                target=100.0, current=0.0, unit="%"
            ),
            GovernanceKPI(
                "Regulatory Audit Pass Rate", "compliance",
                target=100.0, current=0.0, unit="%"
            ),
            GovernanceKPI(
                "Employee Training Completion", "compliance",
                target=95.0, current=0.0, unit="%"
            ),
            GovernanceKPI(
                "Model Performance SLA", "performance",
                target=99.0, current=0.0, unit="%"
            ),
            GovernanceKPI(
                "Fairness Score", "performance",
                target=90.0, current=0.0, unit="%"
            ),
            GovernanceKPI(
                "Explainability Coverage", "performance",
                target=100.0, current=0.0, unit="%"
            ),
            GovernanceKPI(
                "Assessment Lead Time", "efficiency",
                target=5.0, current=0.0,
                unit="business_days",
                direction="lower_is_better"
            ),
            GovernanceKPI(
                "Incident Response Time", "efficiency",
                target=4.0, current=0.0,
                unit="hours", direction="lower_is_better"
            ),
        ]
        for kpi in standard_kpis:
            self.kpis[kpi.name] = kpi

    def update_kpi(self, name: str, value: float):
        """Update a KPI with new measurement"""
        if name in self.kpis:
            self.kpis[name].current = value
            self.history.append({
                "kpi": name,
                "value": value,
                "timestamp": datetime.now().isoformat(),
                "status": self.kpis[name].status.value,
            })

    def get_dashboard_summary(self) -> Dict:
        """Generate dashboard summary"""
        categories = {}
        for kpi in self.kpis.values():
            if kpi.category not in categories:
                categories[kpi.category] = {
                    "total": 0, "on_track": 0,
                    "at_risk": 0, "off_track": 0,
                }
            cat = categories[kpi.category]
            cat["total"] += 1
            cat[kpi.status.value] += 1

        overall_on_track = sum(
            1 for k in self.kpis.values()
            if k.status == KPIStatus.ON_TRACK
        )
        total = len(self.kpis)

        return {
            "organization": self.organization,
            "timestamp": datetime.now().isoformat(),
            "overall_health": f"{overall_on_track}/{total} on track",
            "overall_score": (
                f"{overall_on_track/total:.0%}" if total else "N/A"
            ),
            "by_category": categories,
            "kpi_details": {
                name: {
                    "current": kpi.current,
                    "target": kpi.target,
                    "achievement": f"{kpi.achievement_rate:.0%}",
                    "status": kpi.status.value,
                    "unit": kpi.unit,
                }
                for name, kpi in self.kpis.items()
            },
        }

    def run_pdca_check(self) -> Dict:
        """Execute PDCA Check phase analysis"""
        self.pdca_cycle_count += 1
        off_track = [
            name for name, kpi in self.kpis.items()
            if kpi.status == KPIStatus.OFF_TRACK
        ]
        at_risk = [
            name for name, kpi in self.kpis.items()
            if kpi.status == KPIStatus.AT_RISK
        ]
        improvement_actions = []
        for name in off_track:
            improvement_actions.append({
                "kpi": name,
                "action": f"Investigate root cause for {name}",
                "priority": "HIGH",
                "deadline": (
                    datetime.now() + timedelta(days=14)
                ).strftime("%Y-%m-%d"),
            })
        for name in at_risk:
            improvement_actions.append({
                "kpi": name,
                "action": f"Monitor and optimize {name}",
                "priority": "MEDIUM",
                "deadline": (
                    datetime.now() + timedelta(days=30)
                ).strftime("%Y-%m-%d"),
            })
        return {
            "pdca_cycle": self.pdca_cycle_count,
            "check_date": datetime.now().isoformat(),
            "kpis_off_track": off_track,
            "kpis_at_risk": at_risk,
            "improvement_actions": improvement_actions,
            "next_review": (
                datetime.now() + timedelta(days=90)
            ).strftime("%Y-%m-%d"),
        }